In [ ]:
# Standard library
import copy
import hashlib
import json
import sys
import importlib.metadata as package_metadata

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

# Numerical computation and original model
import numpy as np
import torch

from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# TFLite / LiteRT execution
import litert_torch

from ai_edge_litert.interpreter import Interpreter, OpResolverType

# Quantization and model export
from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e,
)
from litert_torch.quantize.pt2e_quantizer import (
    PT2EQuantizer,
    get_symmetric_quantization_config,
)
from litert_torch.quantize.quant_config import QuantConfig

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Imports loaded.")

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1791885956.098018    5937 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W1013 18:05:56.884000 5937 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Python: /home/randy/Documents/ACompArc/model/.venv/bin/python
PyTorch: 2.13.0+cu130
NumPy: 2.5.3
Imports loaded.


# Load Model

In [5]:
MODEL_ID = "roneneldan/TinyStories-3M"
MODEL_REVISION = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

model_fp32 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float32,
)

model_fp32 = model_fp32.cpu()
model_fp32.eval()

print("Model class:", type(model_fp32).__name__)
print("Parameter dtype:", next(model_fp32.parameters()).dtype)
print("Device:", next(model_fp32.parameters()).device)

W1013 18:07:03.790000 5937 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 53329.98it/s]
[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-3M
Key                                                               | Status     |  | 
------------------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3, 4, 5, 6, 7}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model class: GPTNeoForCausalLM
Parameter dtype: torch.float32
Device: cpu


# Test Model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

prompt = "Once upon a time, a little girl found a tiny dragon in her garden."

inputs = tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
).to(model_fp32.device)

with torch.no_grad():
    generated_ids = model_fp32.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("Input token count:", inputs["input_ids"].shape[1])
print("\nStory:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

Input token count: 16

Story:
Once upon a time, a little girl found a tiny dragon in her garden. She was so excited and wanted to show it to her mom.

The little girl ran to her mom and said, "Mommy, look! A dragon!"

Her mom smiled and said, "Yes, it's very big and strong. But be careful, it's very dangerous."

The little girl nodded and said, "I will be careful. I will be careful


# TinyStories-3M Quantization — Fixed Specification

## Model

| Setting | Value |
|---|---|
| Hugging Face model | `roneneldan/TinyStories-3M` |
| Revision | `cfaf26ec85ecdfc1bd7c2638104cce55cb67f894` |
| Architecture | GPT-Neo |
| Transformer blocks | 8 |
| Hidden size | 128 |
| Attention heads | 16 |
| Head dimension | 8 |
| MLP dimensions | 128 → 512 → 128 |
| Reference model | Pretrained FP32 model, CPU, evaluation mode |

## Input and Generation

| Setting | Value |
|---|---|
| Batch size | 1 |
| Input sequence length | 32 tokens |
| Tokenization | Matching tokenizer; `add_special_tokens=False` |
| Input selection | First 32 token IDs; prompt must contain at least 32 tokens |
| Position IDs | 0–31 |
| Block-0 input | Token embeddings + position embeddings; dropout inactive |
| Block input/output shape | `[1, 32, 128]` |
| Generation limit | 20 additional tokens; may stop earlier at EOS |
| Decoding | Greedy: `do_sample=False`, `num_beams=1` |
| Generation cache | Enabled |
| Maximum resulting sequence length | 52 tokens |


In [7]:
# Input specification
BATCH_SIZE = 1
SEQUENCE_LENGTH = 32
GENERATED_LENGTH = 20

PROMPT = (
    "Once upon a time, a curious little girl named Lily lived in a quiet "
    "village beside a green forest. Every morning, she carried fresh bread "
    "to her grandmother and watched the birds in the trees."
)

# Verify the loaded model's architecture.
config = model_fp32.config

assert config.num_layers == 8
assert config.hidden_size == 128
assert config.num_heads == 16

HEAD_DIM = config.hidden_size // config.num_heads
MLP_HIDDEN_SIZE = config.intermediate_size or 4 * config.hidden_size

assert HEAD_DIM == 8
assert MLP_HIDDEN_SIZE == 512

BLOCK_INPUT_SHAPE = (
    BATCH_SIZE,
    SEQUENCE_LENGTH,
    config.hidden_size,
)

# Choices to apply when we construct the quantizer.
QUANTIZATION_SETTINGS = {
    "scheme": "W8A8",
    "is_per_channel": False,
    "is_dynamic": False,
    "is_qat": False,
}

# Used later for full-model generation tests.
GENERATION_SETTINGS = {
    "max_new_tokens": GENERATED_LENGTH,
    "do_sample": False,
    "num_beams": 1,
    "use_cache": True,
}

print("Block input shape:", BLOCK_INPUT_SHAPE)
print("Head dimension:", HEAD_DIM)
print("MLP hidden size:", MLP_HIDDEN_SIZE)
print("Quantization settings:", QUANTIZATION_SETTINGS)

Block input shape: (1, 32, 128)
Head dimension: 8
MLP hidden size: 512
Quantization settings: {'scheme': 'W8A8', 'is_per_channel': False, 'is_dynamic': False, 'is_qat': False}


# Saving FP32 Model From TinyStories - 3M


In [ ]:
FP32_MODEL_DIR = (
    Path.home()
    / "Documents/ACompArc/model/project/artifacts/models/tinystories_3m_fp32"
)

# Check that the model parameters are still FP32.
assert all(p.dtype == torch.float32 for p in model_fp32.parameters())

# Preserve any previously saved copy.
FP32_MODEL_DIR.mkdir(parents=True, exist_ok=False)

model_fp32.save_pretrained(FP32_MODEL_DIR)
tokenizer.save_pretrained(FP32_MODEL_DIR)

metadata = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "dtype": "float32",
    "quantized": False,
}

(FP32_MODEL_DIR / "source_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)

print("Saved FP32 model and tokenizer to:")
print(FP32_MODEL_DIR)

print("\nSaved files:")
for path in sorted(FP32_MODEL_DIR.iterdir()):
    print(path.name)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.90it/s]

Saved FP32 model and tokenizer to:
/home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_fp32

Saved files:
config.json
generation_config.json
model.safetensors
source_metadata.json
tokenizer.json
tokenizer_config.json


In [ ]:
# LOAD IT
# from pathlib import Path
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# FP32_MODEL_DIR = Path(
#     "/home/randy/Documents/ACompArc/model/project/"
#     "artifacts/models/tinystories_3m_fp32"
# )

# # Load the saved FP32 model.
# model_fp32 = AutoModelForCausalLM.from_pretrained(
#     FP32_MODEL_DIR,
#     local_files_only=True,
#     dtype=torch.float32,
# )
# model_fp32 = model_fp32.cpu()
# model_fp32.eval()

# # Load its matching tokenizer.
# tokenizer = AutoTokenizer.from_pretrained(
#     FP32_MODEL_DIR,
#     local_files_only=True,
# )

# print("Model:", type(model_fp32).__name__)
# print("Weight dtype:", next(model_fp32.parameters()).dtype)
# print("Device:", next(model_fp32.parameters()).device)
# print("Transformer blocks:", len(model_fp32.transformer.h))

# Divide each model per block

In [ ]:
model_fp32.eval()

assert len(model_fp32.transformer.h) == 8

# Independent copies, indexed from block 0 through block 7.
blocks_fp32 = [
    copy.deepcopy(block).cpu().eval()
    for block in model_fp32.transformer.h
]

# Convenient name for the block we will study first.
block0_fp32 = blocks_fp32[0]

print("Extracted Transformer blocks:", len(blocks_fp32))

for i, block in enumerate(blocks_fp32):
    parameter_count = sum(p.numel() for p in block.parameters())
    print(f"Block {i}: {parameter_count:,} parameters")

print("\nComponents inside block 0:")
for name, component in block0_fp32.named_children():
    print(f"  {name}: {type(component).__name__}")

Extracted Transformer blocks: 8
Block 0: 197,888 parameters
Block 1: 197,888 parameters
Block 2: 197,888 parameters
Block 3: 197,888 parameters
Block 4: 197,888 parameters
Block 5: 197,888 parameters
Block 6: 197,888 parameters
Block 7: 197,888 parameters

Components inside block 0:
  ln_1: LayerNorm
  attn: GPTNeoAttention
  ln_2: LayerNorm
  mlp: GPTNeoMLP


In [6]:
print(block0_fp32)

GPTNeoBlock(
  (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (attn): GPTNeoAttention(
    (attention): GPTNeoSelfAttention(
      (attn_dropout): Dropout(p=0.0, inplace=False)
      (resid_dropout): Dropout(p=0.0, inplace=False)
      (k_proj): Linear(in_features=128, out_features=128, bias=False)
      (v_proj): Linear(in_features=128, out_features=128, bias=False)
      (q_proj): Linear(in_features=128, out_features=128, bias=False)
      (out_proj): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (mlp): GPTNeoMLP(
    (c_fc): Linear(in_features=128, out_features=512, bias=True)
    (c_proj): Linear(in_features=512, out_features=128, bias=True)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


# Quantization Block 0 - 7 

## Preparation

In [9]:
# Working copy for export; the original model stays available.
export_model_fp32 = copy.deepcopy(model_fp32).cpu().eval()
export_model_fp32.set_attn_implementation("eager")

assert len(export_model_fp32.transformer.h) == 8
assert BATCH_SIZE == 1
assert SEQUENCE_LENGTH == 32

def collect_block_io(input_ids):
    """Capture the actual FP32 input and output of every block."""
    captured = {}
    handles = []

    def make_hook(block_index):
        def hook(module, args, output):
            hidden_output = (
                output[0] if isinstance(output, (tuple, list)) else output
            )
            captured[block_index] = (
                args[0].detach().clone(),
                hidden_output.detach().clone(),
            )
        return hook

    try:
        for i, block in enumerate(export_model_fp32.transformer.h):
            handles.append(block.register_forward_hook(make_hook(i)))

        with torch.no_grad():
            export_model_fp32.transformer(
                input_ids=input_ids,
                use_cache=False,
                return_dict=True,
            )
    finally:
        for handle in handles:
            handle.remove()

    assert len(captured) == 8
    return [captured[i] for i in range(8)]

# Fixed benchmark prompt from our specification cell.
benchmark_ids = tokenizer(
    PROMPT,
    add_special_tokens=False,
    return_tensors="pt",
)["input_ids"]

assert benchmark_ids.shape[1] >= SEQUENCE_LENGTH
benchmark_ids = benchmark_ids[:, :SEQUENCE_LENGTH]
benchmark_block_io = collect_block_io(benchmark_ids)

# Small initial calibration corpus.
calibration_prompts = [
    "Once upon a time, a little girl named Lily lived beside a quiet "
    "forest. Every morning she carried bread to her grandmother and "
    "greeted all the animals along the path.",

    "Tom found a small red ball underneath his bed. He took it outside "
    "and played with his dog until the sun disappeared behind the hills.",

    "A young rabbit wanted to cross the river, but the water was too "
    "deep. His friends gathered branches and helped him build a bridge.",

    "Mia planted a tiny seed in the garden. She watered it every day, "
    "and soon a tall sunflower grew beside her bedroom window.",

    "The little bird could not fly as high as the others. She practiced "
    "every afternoon until she finally reached the top of the old tree.",

    "Ben heard a strange sound inside the kitchen. He opened the door "
    "slowly and discovered a hungry kitten looking for some food.",

    "A kind farmer shared his vegetables with the people in his village. "
    "When winter arrived, everyone helped him repair his wooden barn.",

    "Sara lost her favorite blue hat at the park. She searched near the "
    "swings and found it beside a sleeping brown puppy.",
]

# Concatenate stories with EOS separators, then take full 32-token chunks.
assert tokenizer.eos_token_id is not None
calibration_tokens = []

for text in calibration_prompts:
    calibration_tokens.extend(
        tokenizer.encode(text, add_special_tokens=False)
    )
    calibration_tokens.append(tokenizer.eos_token_id)

calibration_ids = [
    torch.tensor(
        [calibration_tokens[start:start + SEQUENCE_LENGTH]],
        dtype=torch.long,
    )
    for start in range(
        0,
        len(calibration_tokens) - SEQUENCE_LENGTH + 1,
        SEQUENCE_LENGTH,
    )
]

assert calibration_ids
calibration_block_inputs = [[] for _ in range(8)]

for ids in calibration_ids:
    block_io = collect_block_io(ids)
    for i, (block_input, _) in enumerate(block_io):
        calibration_block_inputs[i].append(block_input)

print("Calibration sequences:", len(calibration_ids))

for i, (block_input, _) in enumerate(benchmark_block_io):
    assert tuple(block_input.shape) == (1, 32, 128)
    print(
        f"Block {i}: input {tuple(block_input.shape)}, "
        f"{len(calibration_block_inputs[i])} calibration samples"
    )

Calibration sequences: 7
Block 0: input (1, 32, 128), 7 calibration samples
Block 1: input (1, 32, 128), 7 calibration samples
Block 2: input (1, 32, 128), 7 calibration samples
Block 3: input (1, 32, 128), 7 calibration samples
Block 4: input (1, 32, 128), 7 calibration samples
Block 5: input (1, 32, 128), 7 calibration samples
Block 6: input (1, 32, 128), 7 calibration samples
Block 7: input (1, 32, 128), 7 calibration samples


## Quantize, Validate, Save

In [10]:
class BlockExportWrapper(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.block = block

    def forward(self, hidden_states):
        output = self.block(
            hidden_states,
            layer_past=None,
            attention_mask=None,
            use_cache=False,
            output_attentions=False,
        )
        return output[0] if isinstance(output, (tuple, list)) else output


def as_numpy(output):
    if isinstance(output, (tuple, list)):
        output = output[0]
    if isinstance(output, torch.Tensor):
        output = output.detach().cpu().numpy()
    return np.asarray(output)


run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

QUANTIZED_BLOCK_DIR = (
    Path.home()
    / "Documents/ACompArc/model/project/artifacts/models"
    / "tinystories_3m_blocks_w8a8"
    / run_name
)
QUANTIZED_BLOCK_DIR.mkdir(parents=True, exist_ok=False)

report = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "sequence_length": SEQUENCE_LENGTH,
    "use_cache": False,
    "attention_implementation": "eager",
    "quantization": {
        "is_per_channel": False,
        "is_dynamic": False,
        "is_qat": False,
        "fold_quantize": False,
        "fully_integer": False,
    },
    "calibration_prompts": calibration_prompts,
    "calibration_sequences": len(calibration_ids),
    "calibration_source": "preceding FP32 blocks",
    "benchmark_prompt": PROMPT,
    "versions": {
        name: package_metadata.version(name)
        for name in [
            "torch", "transformers", "torchao",
            "litert-torch", "ai-edge-litert", "numpy",
        ]
    },
    "blocks": [],
}

# Preserve the token sequences and per-block benchmark references.
validation_data = {
    "benchmark_input_ids": benchmark_ids.numpy(),
    "calibration_input_ids": torch.cat(calibration_ids).numpy(),
}
for i, (x, y) in enumerate(benchmark_block_io):
    validation_data[f"block{i}_input_fp32"] = x.numpy()
    validation_data[f"block{i}_output_fp32"] = y.numpy()

np.savez(
    QUANTIZED_BLOCK_DIR / "validation_inputs.npz",
    **validation_data,
)

print("Saving to:", QUANTIZED_BLOCK_DIR, flush=True)

for i in range(8):
    print(f"\nQuantizing block {i}...", flush=True)

    wrapper = BlockExportWrapper(
        copy.deepcopy(export_model_fp32.transformer.h[i])
    ).eval()

    example_input, native_reference = benchmark_block_io[i]
    sample_args = (example_input,)

    # Check that isolated execution matches the block in the full model.
    with torch.no_grad():
        isolated_reference = wrapper(example_input)

    torch.testing.assert_close(
        isolated_reference,
        native_reference,
        atol=1e-5,
        rtol=1e-5,
    )

    exported = torch.export.export(wrapper, sample_args)
    with torch.no_grad():
        torch.testing.assert_close(
            exported.module()(example_input),
            isolated_reference,
        )

    quantizer = PT2EQuantizer().set_global(
        get_symmetric_quantization_config(
            is_per_channel=False,
            is_dynamic=False,
            is_qat=False,
        )
    )

    prepared = prepare_pt2e(exported.module(), quantizer)

    with torch.no_grad():
        for calibration_input in calibration_block_inputs[i]:
            prepared(calibration_input)

    quantized = convert_pt2e(prepared, fold_quantize=False)

    edge_block = litert_torch.convert(
        quantized,
        sample_args,
        quant_config=QuantConfig(pt2e_quantizer=quantizer),
    )

    output_before_save = as_numpy(edge_block(example_input))
    model_path = QUANTIZED_BLOCK_DIR / f"block{i}_w8a8.tflite"
    edge_block.export(str(model_path))

    # Verify that the saved file reloads with the same output.
    reloaded = litert_torch.load(str(model_path))
    output_after_save = as_numpy(reloaded(example_input))

    np.testing.assert_allclose(
        output_after_save,
        output_before_save,
        atol=1e-6,
        rtol=1e-6,
    )

    error = np.abs(
        output_after_save - native_reference.numpy()
    )
    assert np.isfinite(output_after_save).all()

    # Inspect the exported tensor types and projection quantization.
    interpreter = Interpreter(
        model_path=str(model_path),
        experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
    )
    interpreter.allocate_tensors()
    tensors = {
        t["index"]: t for t in interpreter.get_tensor_details()
    }
    dtype_counts = dict(Counter(
        np.dtype(t["dtype"]).name for t in tensors.values()
    ))

    int8_fc_count = 0
    for op in interpreter._get_ops_details():
        if op["op_name"] == "FULLY_CONNECTED":
            indices = [
                int(op["inputs"][0]),
                int(op["inputs"][1]),
                int(op["outputs"][0]),
            ]
            if all(
                tensors[index]["dtype"] == np.int8
                for index in indices
            ):
                int8_fc_count += 1

    assert int8_fc_count == 6, (
        f"Block {i}: expected six INT8 projections; "
        f"found {int8_fc_count}. Inspect this export."
    )

    block_report = {
        "block_index": i,
        "file": model_path.name,
        "sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
        "bytes": model_path.stat().st_size,
        "int8_fully_connected_ops": int8_fc_count,
        "tensor_dtype_counts": dtype_counts,
        "benchmark_max_abs_error": float(error.max()),
        "benchmark_mean_abs_error": float(error.mean()),
        "reload_check": "passed",
    }
    report["blocks"].append(block_report)

    (QUANTIZED_BLOCK_DIR / "export_report.json").write_text(
        json.dumps(report, indent=2) + "\n"
    )

    print(
        f"Saved {model_path.name} | INT8 FCs: {int8_fc_count} | "
        f"max error: {error.max():.6f} | "
        f"mean error: {error.mean():.6f}",
        flush=True,
    )
    print("Tensor types:", dtype_counts, flush=True)

print("\nSaved all eight blocks to:")
print(QUANTIZED_BLOCK_DIR)

Saving to: /home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_blocks_w8a8/20261013T100746756260Z

Quantizing block 0...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

... (216 outputs left out)

(00:00) [START] Write Model to Bytes

I0000 00:00:1791886082.863653    5937 flatbuffer_export.cc:4342] Estimated count of arithmetic ops: 14.844 M  ops, equivalently 7.422 M  MACs


(00:00) [ DONE] Write Model to Bytes (+00:00)

Saved block7_w8a8.tflite | INT8 FCs: 6 | max error: 0.011993 | mean error: 0.002239
Tensor types: {'float32': 64, 'int32': 22, 'bool': 1, 'int8': 37}

Saved all eight blocks to:
/home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_blocks_w8a8/20261013T100746756260Z
